# Week 3 — Data Engineering for AI: Pipelines, Lineage, and Workflows
**Project:** MedLabel: AI-Based Multi-Prescription Recognition and Medication Timetable Generation  
**Modality:** Computer Vision (Prescription Images — Pathway C) + Medication Master Reference Table  
**Course:** SAIA3113 — AI Engineering  

## Cell 1 — Workspace Setup (RUN FIRST)
Initialises the standard Colab workspace directories.

In [1]:
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

## Cell 3 — Verify Uploaded Inputs
Inspects files uploaded into `data/raw` recursively.

In [2]:
from pathlib import Path

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Proceeding to metadata utilities.")

## Cell 4 — Source Selection + Lineage Utilities
Provides `select_source()`, cryptographic `source_hash()`, and manifest saving.

In [3]:
import json
import hashlib
from datetime import datetime, timezone

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file() and not selected.is_dir():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    if len(matches) != 1:
        raise ValueError("Select exactly one source.")
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return manifest

print("Lineage utilities initialized.")

## Cell 4A — Common Lineage Registry

In [4]:
lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print("Registered:", source.name, "->", output.name)

print("Common lineage registry ready.")

## Cell 5C — Pathway C: Computer Vision / Prescription Images
Inspects image properties, validates dimensions, and standardizes to 512×512 RGB.

In [5]:
from PIL import Image, UnidentifiedImageError
import pandas as pd

source_dir = RAW_DIR / "images"
image_files = sorted(p for p in source_dir.iterdir() if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"})
assert image_files, "No images found in data/raw/images"

print("--- Source Images ---")
print("Count:", len(image_files))
print("Sample files:", [p.name for p in image_files[:3]])

# Step 3: Configure preparation settings
# 512x512 chosen over 224x224 to preserve fine handwriting and decimal point details
TARGET_SIZE = (512, 512)
CONVERT_TO_RGB = True

image_dir = PROCESSED_DIR / "images"
image_dir.mkdir(parents=True, exist_ok=True)
rows = []

for f in image_files:
    with Image.open(f) as im:
        im.load()
        original_size = im.size
        original_mode = im.mode
        transformed = im.convert("RGB") if CONVERT_TO_RGB else im.copy()
        transformed = transformed.resize(TARGET_SIZE, Image.Resampling.LANCZOS)
        target = image_dir / f"{f.stem}_{source_hash(f)[:10]}.png"
        transformed.save(target, "PNG")
        rows.append({
            "source": str(f.relative_to(RAW_DIR)),
            "source_sha256": source_hash(f),
            "original_width": original_size[0],
            "original_height": original_size[1],
            "original_mode": original_mode,
            "output": str(target.relative_to(BASE_DIR)),
            "output_width": TARGET_SIZE[0],
            "output_height": TARGET_SIZE[1],
            "processing": f"convert_to_rgb={CONVERT_TO_RGB}; resize={TARGET_SIZE}"
        })

prepared_path = PROCESSED_DIR / "image_manifest.csv"
pd.DataFrame(rows).to_csv(prepared_path, index=False)
print("\nPATHWAY C completed successfully. Manifest:", prepared_path)

## Step B3 — Basic Output Contract Check

In [6]:
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:', prepared_path, 'Size:', prepared_path.stat().st_size, 'bytes')
print('Validation: PASS - Output exists and is non-empty.')

## Cell 6A & 6B — Register Lineage & Generate Common Manifest

In [7]:
# Register each image in lineage
for row in rows:
    register_lineage(
        source=RAW_DIR / row["source"],
        output=BASE_DIR / row["output"],
        processing=row["processing"]
    )

lineage_records = []
for i, entry in enumerate(lineage_entries.values(), 1):
    src = entry["source"]
    out = entry["output"]
    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(src.relative_to(RAW_DIR)),
        "source_sha256": source_hash(src),
        "prepared_output": str(out.relative_to(BASE_DIR)),
        "prepared_sha256": source_hash(out),
        "processing_note": entry["processing"],
        "source_version": "Mendeley Prescription Dataset v2.0",
        "access_rights": "CC BY 4.0 Open Access",
        "status": "Prepared; quality review pending"
    })

manifest = save_manifest(lineage_records)
print("Common lineage manifest generated with", len(manifest["records"]), "records.")

## Cell 7 — Common V1 Pipeline Utilities
Defines the reusable end-to-end execution controller.

In [8]:
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_RUNS = {}

def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if folder and not source.is_dir():
        raise FileNotFoundError(f"Folder not found: {source}")
    if not folder and not source.is_file():
        raise FileNotFoundError(f"File not found: {source}")
    return source

def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    source = Path(source)
    inputs = sorted(p for p in source.iterdir() if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"})
    inspection = inspect_fn(inputs, configuration)
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)
    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]
    validation = {
        "output_count": len(outputs),
        "output_sha256": [source_hash(p) for p in outputs],
        "output_bytes": [p.stat().st_size for p in outputs]
    }
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [{"path": str(p), "sha256": source_hash(p)} for p in inputs],
        "outputs": [str(p) for p in outputs],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Mendeley Dataset v2.0",
        "access_rights": "CC BY 4.0 Open Access"
    }
    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    return record

def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    def runner():
        return execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first

print("Cell 7 ready.")

## Cell 8C — Execute V1 Image Data Pipeline (Pathway C)

In [9]:
SOURCE_C = resolve_source("images", folder=True)
CONFIG_C = {"resize": [512, 512], "convert_rgb": True}

def inspect_images(inputs, cfg):
    properties = []
    for src in inputs:
        with Image.open(src) as im:
            properties.append({"name": src.name, "size": list(im.size), "mode": im.mode})
    return {"image_count": len(inputs), "sample": properties[:3]}

def prepare_images(inputs, out, cfg):
    outputs = []
    mapping = []
    image_size = cfg["resize"]
    convert_rgb = cfg["convert_rgb"]
    for i, src in enumerate(inputs, 1):
        with Image.open(src) as im:
            img = im.convert("RGB") if convert_rgb else im.copy()
            if image_size:
                img = img.resize(tuple(image_size), Image.Resampling.LANCZOS)
            target = out / f"prepared_{i:03d}.png"
            img.save(target, "PNG")
            outputs.append(target)
            mapping.append({"source": str(src), "output": str(target)})
    return outputs, mapping, f"convert_rgb={convert_rgb}; resize={image_size}"

run_C = register_and_run("images", SOURCE_C, inspect_images, prepare_images, CONFIG_C)
print("Cell 8C completed successfully.")

## Cell 9 — Repeatability and Provenance Verification

In [10]:
assert PIPELINE_RUNS, "No pipeline runs registered."
print("--- Repeatability Verification ---")
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    second = entry["runner"]()
    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    assert first["source_output_mapping"] == second["source_output_mapping"], f"Mapping changed for {kind}"
    print(f"PASS: {kind} — source fingerprints, prepared outputs and mappings match identically.")

print("\nAll selected pipelines passed repeatability smoke checks.")